In [5]:
import pandas as pd

file_path = r"C:\ctd_data\snacks_large_sample.csv"
df = pd.read_csv(file_path,encoding="latin1")  # or try "utf-8"

# Fix encoding issues
df["ingredients_text"] = (
    df["ingredients_text"]
    .str.encode("latin1")
    .str.decode("utf-8")
)

# Save clean file
df.to_csv("snacks_cleaned.csv", index=False, encoding="utf-8")

In [ ]:
import pandas as pd
import unicodedata

# ================================
# STEP 1: LOAD DATA (safe encoding)
# ================================
file_path = r"C:\ctd_data\snacks_large_sample.csv"

df = pd.read_csv(
    file_path,
    encoding="latin1",   # most likely correct for your case
    low_memory=False
)

print("Loaded rows:", len(df))


# ================================
# STEP 2: FIX ENCODING 
# ================================
def fix_encoding(text):
    if pd.isna(text):
        return text
    try:
        fixed = text.encode("latin1").decode("utf-8")
        return fixed
    except:
        return text  # keep original if already correct

df["ingredients_text"] = df["ingredients_text"].apply(fix_encoding)


# ================================
# STEP 3: LOWERCASE (consistency)
# ================================
df["ingredients_text"] = df["ingredients_text"].str.lower()


# ================================
# STEP 4: REMOVE ACCENTS
# ================================
def remove_accents(text):
    if pd.isna(text):
        return text
    return ''.join(
        c for c in unicodedata.normalize('NFKD', text)
        if not unicodedata.combining(c)
    )

df["ingredients_text"] = df["ingredients_text"].apply(remove_accents)


# ================================
# STEP 5: BASIC CLEANING
# ================================
# remove weird characters, extra spaces
df["ingredients_text"] = (
    df["ingredients_text"]
    .str.replace(r"[^a-z0-9,:\-\s\(\)]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


# ================================
# STEP 6: OPTIONAL NORMALIZATION (SMART)
# ================================
# Minimal dictionary (expand later if needed)
mapping = {
    "sucre": "sugar",
    "sel": "salt",
    "aromes": "flavor",
    "lecithine": "lecithin",
    "dextrose": "dextrose",
    "huile": "oil",
    "farine": "flour"
}

def normalize_terms(text):
    if pd.isna(text):
        return text
    for k, v in mapping.items():
        text = text.replace(k, v)
    return text

df["ingredients_cleaned"] = df["ingredients_text"].apply(normalize_terms)


# ================================
# STEP 7: CREATE INGREDIENT LIST
# ================================
df["ingredients_list"] = df["ingredients_cleaned"].str.split(",")


# ================================
# STEP 8: REMOVE EMPTY ENTRIES
# ================================
df["ingredients_list"] = df["ingredients_list"].apply(
    lambda x: [i.strip() for i in x if isinstance(i, str) and i.strip() != ""]
)


# ================================
# STEP 9: QUICK VALIDATION
# ================================
print("\nSample cleaned ingredients:\n")
print(df["ingredients_cleaned"].head(5))


# ================================
# STEP 10: SAVE CLEAN FILE
# ================================
output_path = r"C:\ctd_data\snacks_cleaned.csv"

df.to_csv(output_path, index=False, encoding="utf-8")

print("\n✅ Cleaned file saved to:", output_path)

Loaded rows: 100000

Sample cleaned ingredients:

0    sirop de glucose, (065), proteines de lait, cr...
1    flour de ble, sugar, graisse de palme, 4,5 noi...
2    pate de taro, maltose, sugar, riz gluant, epai...
3    chocolat noir (min57 cacao) 50 (masse de cacao...
4    fourrage gout cacao noisette 35 (sugar de cann...
Name: ingredients_cleaned, dtype: str

✅ Cleaned file saved to: C:\ctd_data\snacks_cleaned.csv
